# B3: fine-tuned encoder baseline (XLM-R), on Kaggle's free GPU

B3 is the project's **published reference baseline**: the MASSIVE paper (FitzGerald et al., ACL 2023) reports XLM-R and mT5 results with released code, so this has a published, reproducible basis. Publication alone does not make it state of the art, and it must be described as "the dataset's published reference baseline" unless the literature check supports a stronger claim.

Two runs, as the plan requires:

- **B3-k** — trained on the same k = 10 examples per intent that every LLM system receives. This is the *fair* supervised comparison.
- **B3-full** — trained on the full official train split for the project's domains. An **upper reference only**; it must never be presented as a matched comparison. A fine-tuned encoder with full data may well beat every few-shot system; if so, that gets reported plainly.

## What this notebook does and does not do

- It trains joint intent classification + slot BIO tagging, like the MASSIVE reference recipe.
- It tunes the out-of-scope threshold on **development data only**, by maximum softmax confidence — the same rule type as B0's TF-IDF threshold.
- It **exports predictions and does not compute final metrics.** Scoring happens back on the laptop with `src/evaluation/scorer.py`, so B3 is scored by exactly the same code as every other system. Metrics computed here with different code would not be comparable.

## Freeze discipline, enforced by construction

The uploaded `b3_splits.json` contains **only the k-shot pools and development splits for P1 and P2**. No test labels and nothing from the held-out project P3 are present, so this notebook *cannot* touch them even by mistake. Test predictions are produced in a separate run after the G10 freeze.

## Setup

1. Kaggle → Create → Notebook. Settings → Accelerator → **GPU T4 x2** (or P100).
2. Add Data → Upload a dataset containing `data/manifests/b3_splits.json` from this repository. Note the path it lands at and set `SPLITS_PATH` below.
3. Internet must be **on** (the notebook downloads MASSIVE 1.1 and the XLM-R weights).
4. Run all. Download `b3_predictions.json` at the end and place it in `runs/` on the laptop.

Free-tier limits to plan around: roughly 30 GPU hours a week, 12-hour sessions.

In [ ]:
# Point this at the uploaded b3_splits.json.
SPLITS_PATH = "/kaggle/input/b3-splits/b3_splits.json"

MODEL_NAME = "xlm-roberta-base"  # the MASSIVE paper's reference encoder family
SEED = 0
EPOCHS_KSHOT = 30   # tiny training set, so more passes
EPOCHS_FULL = 5
LEARNING_RATE = 3e-5
BATCH_SIZE = 16
MAX_LENGTH = 64

import json, os, random, hashlib, re, tarfile, urllib.request
from pathlib import Path

import numpy as np
import torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [ ]:
# --- MASSIVE 1.1, verified against the hash recorded at G2 ---
URL = "https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.1.tar.gz"
EXPECTED_ARCHIVE_SHA256 = "4cba5faa11c71437928e17cb1b9b3d8b8e727e7ea363a3a9a8045e19c0491577"

archive = Path("massive-1.1.tar.gz")
if not archive.exists():
    urllib.request.urlretrieve(URL, archive)

digest = hashlib.sha256(archive.read_bytes()).hexdigest()
assert digest == EXPECTED_ARCHIVE_SHA256, (
    f"archive hash {digest} does not match the one recorded at G2. "
    "The dataset release changed; stop and investigate rather than training on different data."
)
print("archive hash matches the G2 record")

with tarfile.open(archive) as tar:
    tar.extract("1.1/data/en-US.jsonl", path=".")

rows = [json.loads(line) for line in open("1.1/data/en-US.jsonl", encoding="utf-8")]
by_id = {f"massive-{r['id']}": r for r in rows}
print(f"{len(rows)} en-US records loaded")

In [ ]:
# --- span conversion, identical to src/evaluation/massive.py ---
SLOT_PATTERN = re.compile(r"\[(?P<type>[^\[\]:]+?)\s*:\s*(?P<value>[^\[\]]+?)\]")

def parse_annotation(utt, annot_utt):
    """Inline [type : value] annotations to character spans, raising on mismatch."""
    entities, parts, cursor, last = [], [], 0, 0
    for m in SLOT_PATTERN.finditer(annot_utt):
        literal = annot_utt[last:m.start()]
        parts.append(literal); cursor += len(literal)
        value = m.group("value").strip()
        entities.append({"type": m.group("type").strip(), "text": value,
                         "start": cursor, "end": cursor + len(value)})
        parts.append(value); cursor += len(value); last = m.end()
    parts.append(annot_utt[last:])
    reconstructed = "".join(parts)
    if reconstructed != utt:
        raise ValueError(f"annotation does not reconstruct {utt!r}")
    return entities

checked = sum(1 for r in rows if parse_annotation(r["utt"], r["annot_utt"]) is not None)
print(f"span conversion verified on {checked} records")

In [ ]:
# --- rebuild the frozen splits from their committed case IDs ---
splits = json.loads(Path(SPLITS_PATH).read_text(encoding="utf-8"))
print(splits["purpose"])
for note in splits["deliberately_excluded"]:
    print(" excluded:", note)

OOS_LABEL = "none_of_the_above"

def build_case(case_id, in_scope):
    row = by_id[case_id]
    entities = parse_annotation(row["utt"], row["annot_utt"])
    return {"case_id": case_id, "utt": row["utt"],
            "true_intent": row["intent"], "scenario": row["scenario"],
            "in_scope": in_scope, "entities": entities if in_scope else [],
            "partition": row["partition"]}

projects = {}
for pid, spec in splits["projects"].items():
    domains = set(spec["domains"])
    kshot = [build_case(cid, True) for cid in spec["case_ids"]["kshot"]]
    dev = [build_case(cid, by_id[cid]["scenario"] in domains) for cid in spec["case_ids"]["dev"]]
    # The full train split for this project's domains: B3-full's training data.
    full = [build_case(f"massive-{r['id']}", True) for r in rows
            if r["scenario"] in domains and r["partition"] == "train"]

    assert len(kshot) == spec["counts"]["kshot"], pid
    assert len(dev) == spec["counts"]["dev_total"], pid
    assert sum(c["in_scope"] for c in dev) == spec["counts"]["dev_in_scope"], pid
    assert {c["partition"] for c in kshot} == {"train"}, pid
    assert {c["partition"] for c in dev} == {"dev"}, pid
    # No out-of-scope example may ever be trained on, in any system.
    assert all(c["in_scope"] for c in kshot + full), pid

    projects[pid] = {"spec": spec, "kshot": kshot, "dev": dev, "full": full}
    print(f"{pid} {spec['name']}: {len(kshot)} k-shot, {len(full)} full-train, "
          f"{len(dev)} dev ({sum(c['in_scope'] for c in dev)} in-scope), "
          f"{len(spec['intents'])} intents, {len(spec['entity_schema'])} entity types")

In [ ]:
# --- character spans to token BIO labels ---
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def bio_labels(entity_types):
    labels = ["O"]
    for t in entity_types:
        labels += [f"B-{t}", f"I-{t}"]
    return labels

def encode(case, intent_to_id, label_to_id):
    enc = tokenizer(case["utt"], truncation=True, max_length=MAX_LENGTH,
                    return_offsets_mapping=True)
    tags = []
    for start, end in enc["offset_mapping"]:
        if start == end:  # special token
            tags.append(-100)
            continue
        tag = "O"
        for entity in case["entities"]:
            if start >= entity["start"] and end <= entity["end"]:
                prefix = "B" if start == entity["start"] else "I"
                candidate = f"{prefix}-{entity['type']}"
                if candidate in label_to_id:
                    tag = candidate
                break
        tags.append(label_to_id[tag])
    enc.pop("offset_mapping")
    enc["slot_labels"] = tags
    # An out-of-scope case never appears in training, so its intent id is unused.
    enc["intent_label"] = intent_to_id.get(case["true_intent"], -100)
    return enc

print("tokenizer ready:", MODEL_NAME)

In [ ]:
# --- joint intent + slot model ---
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel

class JointModel(nn.Module):
    """One encoder, two heads: intent over [CLS], slots per token."""

    def __init__(self, n_intents, n_slots):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.intent_head = nn.Linear(hidden, n_intents)
        self.slot_head = nn.Linear(hidden, n_slots)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        pooled = self.dropout(out[:, 0])
        return self.intent_head(pooled), self.slot_head(self.dropout(out))

class CaseDataset(Dataset):
    def __init__(self, cases, intent_to_id, label_to_id):
        self.items = [encode(c, intent_to_id, label_to_id) for c in cases]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]

def collate(batch):
    width = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    return {
        "input_ids": torch.tensor([b["input_ids"] + [pad] * (width - len(b["input_ids"])) for b in batch]),
        "attention_mask": torch.tensor([b["attention_mask"] + [0] * (width - len(b["attention_mask"])) for b in batch]),
        "slot_labels": torch.tensor([b["slot_labels"] + [-100] * (width - len(b["slot_labels"])) for b in batch]),
        "intent_label": torch.tensor([b["intent_label"] for b in batch]),
    }

print("model classes defined")

In [ ]:
# --- training and prediction ---
device = "cuda" if torch.cuda.is_available() else "cpu"

def train(cases, intents, entity_types, epochs):
    set_seed(SEED)
    intent_to_id = {name: i for i, name in enumerate(intents)}
    labels = bio_labels(entity_types)
    label_to_id = {name: i for i, name in enumerate(labels)}

    model = JointModel(len(intents), len(labels)).to(device)
    loader = DataLoader(CaseDataset(cases, intent_to_id, label_to_id),
                        batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
    optimiser = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    model.train()
    for epoch in range(epochs):
        total = 0.0
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimiser.zero_grad()
            intent_logits, slot_logits = model(batch["input_ids"], batch["attention_mask"])
            loss = (loss_fn(intent_logits, batch["intent_label"])
                    + loss_fn(slot_logits.reshape(-1, len(labels)), batch["slot_labels"].reshape(-1)))
            loss.backward(); optimiser.step()
            total += loss.item()
        if epoch % 5 == 0 or epoch == epochs - 1:
            print(f"  epoch {epoch + 1}/{epochs} loss {total / max(1, len(loader)):.4f}")
    return model, intents, labels

@torch.no_grad()
def predict(model, cases, intents, labels):
    """Closed-set intent, its softmax confidence, and BIO-decoded entity strings.

    Entities are emitted as the exact substring of the utterance the predicted
    span covers, so the laptop's shared span rule locates them. B3 is therefore
    scored through the same evaluator as every other system.
    """
    model.eval()
    results = []
    for case in cases:
        enc = tokenizer(case["utt"], truncation=True, max_length=MAX_LENGTH,
                        return_offsets_mapping=True, return_tensors="pt")
        offsets = enc.pop("offset_mapping")[0].tolist()
        enc = {k: v.to(device) for k, v in enc.items()}
        intent_logits, slot_logits = model(enc["input_ids"], enc["attention_mask"])
        probabilities = torch.softmax(intent_logits[0], dim=-1)
        best = int(probabilities.argmax())
        slot_ids = slot_logits[0].argmax(dim=-1).tolist()

        entities, current = [], None
        for (start, end), slot_id in zip(offsets, slot_ids):
            if start == end:
                continue
            tag = labels[slot_id]
            if tag == "O":
                current = None
                continue
            prefix, _, entity_type = tag.partition("-")
            if prefix == "B" or current is None or current["type"] != entity_type:
                current = {"type": entity_type, "start": start, "end": end}
                entities.append(current)
            else:
                current["end"] = end
        results.append({
            "case_id": case["case_id"],
            "closed_set_intent": intents[best],
            "confidence": float(probabilities[best]),
            "entities": [{"type": e["type"], "text": case["utt"][e["start"]:e["end"]]}
                         for e in entities],
        })
    return results

print(f"device: {device}")

In [ ]:
# --- run B3-k and B3-full for each development project ---
output = {
    "model_name": MODEL_NAME,
    "seed": SEED,
    "hyperparameters": {"lr": LEARNING_RATE, "batch_size": BATCH_SIZE,
                        "max_length": MAX_LENGTH,
                        "epochs_kshot": EPOCHS_KSHOT, "epochs_full": EPOCHS_FULL},
    "torch_version": torch.__version__,
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "split": "dev",
    "note": ("Predictions only. Metrics are computed on the laptop with "
             "src/evaluation/scorer.py so B3 is scored by the same code as every "
             "other system. Out-of-scope thresholds are tuned there, on dev only."),
    "runs": {},
}

for pid, data in projects.items():
    intents = data["spec"]["intents"]
    entity_types = data["spec"]["entity_schema"]
    for variant, training_cases, epochs in (
        ("B3-k", data["kshot"], EPOCHS_KSHOT),
        ("B3-full", data["full"], EPOCHS_FULL),
    ):
        print(f"\n=== {pid} {variant}: {len(training_cases)} training cases ===")
        model, used_intents, labels = train(training_cases, intents, entity_types, epochs)
        predictions = predict(model, data["dev"], used_intents, labels)
        output["runs"][f"{pid}|{variant}"] = {
            "project": pid,
            "variant": variant,
            "n_training_cases": len(training_cases),
            "epochs": epochs,
            "intents": used_intents,
            "entity_schema": entity_types,
            "predictions": predictions,
        }
        del model
        torch.cuda.empty_cache()
        print(f"  {len(predictions)} development predictions recorded")

Path("b3_predictions.json").write_text(json.dumps(output, indent=1), encoding="utf-8")
print("\nwrote b3_predictions.json")
print("Download it and run: python scripts/score_b3.py --predictions b3_predictions.json")

## After this notebook

1. Download `b3_predictions.json` from the Kaggle output panel.
2. On the laptop: `python scripts/score_b3.py --predictions b3_predictions.json`

That script tunes the out-of-scope threshold on development data by the same documented rule B0 uses, applies the shared span rule and scorer, and writes a proper run folder per variant.

## What to record for the paper

- The exact `MODEL_NAME`, hyperparameters, seed and GPU reported above.
- That **B3-k** is the matched comparison and **B3-full** is an upper reference only.
- That training used no out-of-scope examples, like every other system.
- That B3's entity output is BIO-decoded to the utterance's own substrings and then scored through the same span rule and scorer as every other system.
- G5 gate condition: confirm the recipe against the official repository (`https://github.com/alexa/massive`) and check the DA1 literature review for any stronger, newer method on MASSIVE before describing B3 as anything more than the dataset's published reference baseline.